# 2. TTL과 만료
ttl - Redis는 키 수명을 주는 방법

Redis는 키마다 수명을 줄 수 있다. 시간이 지나면 스스로 사라진다.
캐시가 오래된 값을 계속 들고 있지 않게 하는 장치다.

위에서부터 셀을 하나씩 실행합니다 (`Shift + Enter`).

TODO 를 채우기 전에는 결과가 비어 있게 나온다 (None, [], {}).
`...` 은 파이썬 문법상 유효해서 오류 없이 지나가기 때문이다.
결과가 비어 있으면 고장난 것이 아니라 아직 안 채운 것이다.

In [1]:
import time

from redis_client import r

## 1. TTL 거는 방법

In [7]:
# 방법 1) 저장하면서 같이 건다. ex 는 초 단위다.
# TODO 1. 3초 뒤 사라지는 키를 만든다.  힌트: r.set(키, 값, ex=초) / 단위는 세컨
r.set("day14:a", "3초 뒤 사라짐", ex=3)
print("a 의 TTL:", r.get("day14:a"))

a 의 TTL: 3초 뒤 사라짐


In [11]:
print("a 의 TTL:", r.get("day14:a"))

a 의 TTL: None


In [14]:

# 방법 2) 키를 먼저 만들고 키 만료 시간은 나중에 걸기
r.set("day14:b", "값")
# TODO 2. 이미 있는 키에 10초 만료를 건다.  힌트: r.expire(키, 초)
r.expire("day14:b", 10)

print("a 의 TTL:", r.ttl("day14:a"))
print("b 의 TTL:", r.ttl("day14:b"))

# 예전 코드에는 setex(키, 초, 값) 도 보이는데, 지금은 위 두 가지를 쓴다.

a 의 TTL: -2
b 의 TTL: 10


## 2. TTL 값의 뜻

In [27]:
r.set("day14:forever", "만료 없음")

print("만료가 있는 키:", r.ttl("day14:b"), "  남은 초")
print("만료가 없는 키:", r.ttl("day14:forever"), "  -1 이면 만료 없음")
print("아예 없는 키:", r.ttl("day14:없는키"), "  -2 이면 키가 없음")

만료가 있는 키: -2   남은 초
만료가 없는 키: -1   -1 이면 만료 없음
아예 없는 키: -2   -2 이면 키가 없음


## 3. 사라지는 것을 직접 보기

In [28]:
# 바로 위에서 건 키는 여기 오기까지 시간이 흘렀으니 새로 건다.
r.set("day14:watch", "3초 뒤 사라짐", ex=3)
print("3초 만료로 걸었다. 1초마다 확인한다.")
print()

for i in range(5):
    ttl = r.ttl("day14:watch")
    value = r.get("day14:watch")
    print(i, "초 경과   TTL:", ttl, "  값:", value)
    time.sleep(1)

print()
print("지우는 코드를 쓰지 않았는데 키가 스스로 사라졌다.")

3초 만료로 걸었다. 1초마다 확인한다.

0 초 경과   TTL: 3   값: 3초 뒤 사라짐
1 초 경과   TTL: 2   값: 3초 뒤 사라짐
2 초 경과   TTL: 1   값: 3초 뒤 사라짐
3 초 경과   TTL: -2   값: None
4 초 경과   TTL: -2   값: None

지우는 코드를 쓰지 않았는데 키가 스스로 사라졌다.


## 4. 가장 많이 틀리는 부분

In [30]:
r.set("day14:d", "값", ex=100)
print("만료 100초로 저장:", r.ttl("day14:d"))

# 만료를 없애고 싶으면 persist 를 쓴다.
# TODO 3. 만료를 없앤다.  힌트: r.persist(키)
...
print("persist 로 만료 제거:", r.ttl("day14:d"))

r.expire("day14:d", 100)
print("다시 100초 걸기:", r.ttl("day14:d"))



만료 100초로 저장: 100
persist 로 만료 제거: 100
다시 100초 걸기: 100


In [31]:
# 여기가 함정이다. 값만 덮어쓰면 만료가 사라진다.
# TODO 4. ex 를 주지 않고 값만 덮어쓴다.  힌트: r.set(키, 값)
r.set("day14:d", "새 값")
print("set 으로 값만 덮어쓰면:", r.ttl("day14:d"), "  만료가 사라졌다")

r.set("day14:d", "새 값", ex=100)
print("ex 를 다시 주면:", r.ttl("day14:d"))

print()
print("캐시를 갱신할 때 ex 를 빠뜨리면 그 키는 영원히 남는다.")

set 으로 값만 덮어쓰면: -1   만료가 사라졌다
ex 를 다시 주면: 100

캐시를 갱신할 때 ex 를 빠뜨리면 그 키는 영원히 남는다.


## 5. TTL 은 얼마로 잡나

In [ ]:
print("정답은 없다. '틀린 값이 보여도 괜찮은 시간'으로 잡는다.")
print()
print("로그인 상태    300초    자주 바뀌지 않는다")
print("대화 이력       30초    대화 중에는 자주 바뀐다")
print("공지사항      3600초    거의 안 바뀐다")
print()
print("짧게 잡으면 최신 값을 보여주지만 DB 를 자주 부른다.")
print("길게 잡으면 빠르지만 오래된 값을 보여줄 수 있다.")
print()
print("TTL 만으로 부족하면 값이 바뀔 때 캐시를 지운다. 다음 실습에서 다룬다.")

## 6. 정리

In [ ]:
keys = r.keys("day14:*")
for key in keys:
    r.delete(key)

print(len(keys), "개 삭제")
print("남은 키 개수:", r.dbsize())